# Liberaries

In [21]:
import glob
import pandas as pd
import os

In [12]:
folder_path = r"data"
years_str=range(2019, 2020)

# Removing Headers

In [13]:
# function to remove the metadata from everyearfile
def remove_first_n_lines(file_path, n=18):
    lines = []
    with open(file_path, "r") as f:
        lines = f.readlines()[n:]
    with open(file_path, "w") as f:
        f.writelines(lines)

In [14]:
# D:/data/{r_name}/{y}/{param}.csv
for file_path in glob.glob(f"{folder_path}/**/*.csv", recursive=True):
    remove_first_n_lines(file_path)

# Append

## append the common in all the years

In [19]:

cols = None 
for year in years_str:
    pattern = os.path.join(folder_path, '**', str(year), '*.csv')
    year_files = glob.glob(pattern, recursive=True)
    year_cols = set()
    for file_path in year_files:
        filename = os.path.splitext(os.path.basename(file_path))[0]
        year_cols.add(filename)

    if cols is None:
        cols = year_cols
    else:
        cols = cols.intersection(year_cols)
cols = list(cols) if cols else []
print(cols)
print(len(cols))


['T2M', 'Z0M', 'ALLSKY_SFC_SW_DWN']
3


In [ ]:
output_dir = os.path.join(folder_path, "data_appended")
os.makedirs(output_dir, exist_ok=True)

for col in cols:
    pattern = os.path.join(folder_path, "**", f"{col}.csv")
    file_paths = glob.glob(pattern, recursive=True)

    if not file_paths:
        print(f"No files found for {col}")
        continue

    dataframes = []
    for file_path in file_paths:
        df = pd.read_csv(file_path)
        dataframes.append(df)
    combined_df = pd.concat(dataframes, ignore_index=True)

    combined_path = os.path.join(output_dir, f"{col}.csv")
    combined_df.to_csv(combined_path, index=False)
    print(f"Saved combined file for {col} with {len(combined_df)} rows.")

    for file_path in file_paths:
        os.remove(file_path)
        print(f"Deleted {file_path}")
    for year in years_str:    
        os.rmdir(f"{folder_path}/large_region/{year}")
        os.rmdir(f"{folder_path}/small_region/{year}")


Saved combined file for T2M with 271932 rows.
Deleted data\large_region\2019\T2M.csv
Deleted data\large_region\2020\T2M.csv
Deleted data\small_region\2019\T2M.csv
Deleted data\small_region\2020\T2M.csv
Saved combined file for Z0M with 271932 rows.
Deleted data\large_region\2019\Z0M.csv
Deleted data\large_region\2020\Z0M.csv
Deleted data\small_region\2019\Z0M.csv
Deleted data\small_region\2020\Z0M.csv
Saved combined file for ALLSKY_SFC_SW_DWN with 89182 rows.
Deleted data\large_region\2019\ALLSKY_SFC_SW_DWN.csv
Deleted data\large_region\2020\ALLSKY_SFC_SW_DWN.csv
Deleted data\small_region\2019\ALLSKY_SFC_SW_DWN.csv
Deleted data\small_region\2020\ALLSKY_SFC_SW_DWN.csv


# Merging files

**Make sure to put all the appended files in the right place before running this**

In [35]:
def Merge():
    df_small=None
    df_large=None
    for cur_file in glob.glob(f"{folder_path}/data_appended/*.csv"):
        test=pd.read_csv(cur_file).reset_index(drop=True)
        
        if df_small is None and (len(test)<679272):
            df_small=test
        elif df_large is None and (len(test)>679272):
            df_large =test
        else :
            if df_small is not None and (len(test) < 679272):
                df_small = pd.merge(df_small, test, how='outer').reset_index(drop=True)
            elif df_large is not None and (len(test) > 679272):
                df_large = pd.merge(df_large, test, how='outer').reset_index(drop=True)
    
    if df_small is not None:
        os.makedirs(f"{folder_path}/data_merged", exist_ok=True)
        df_small.to_csv(f"{folder_path}/data_merged/df_small.csv", index=False)
    if df_large is not None:
        os.makedirs(f"{folder_path}/data_merged", exist_ok=True)
        df_large.to_csv(f"{folder_path}/data_merged/df_large.csv", index=False)
    for cur_file in glob.glob(f"{folder_path}/data_appended/*.csv"):    
        os.remove(cur_file) 
    os.rmdir(f"{folder_path}/data_appended")    

In [36]:
Merge()